In [0]:
use catalog dev_l2;

In [0]:
-- Q01
-- overall KPI summary, net revenue, orders, active customers, average order value
select
round(sum(net_price),2) as total_revenue,
count(distinct order_key) as total_orders,
count(distinct customer_key) as active_customer,
avg(total_price) as avg_order_value
from gold.fact_sales

In [0]:
-- Q02
-- monthly revenue trend across all years
select
year(order_date) as year,
month(order_date) as month,
sum(net_price) as monthly_revenue
from gold.fact_sales
group by year, month
order by year, month


In [0]:
-- Q03 Annual revenue with year over year change
select
year(order_date) as year,
sum(net_price) as annual_revenue,
lag(sum(net_price)) over (order by year(order_date)) as previous_year_revenue,
round(sum(net_price) - lag(sum(net_price)) over (order by year(order_date)),2) as yoy_change
from gold.fact_sales
group by year
order by year

In [0]:
-- Q04 Revenue by customer market segment
select
c.market_segment as market_segment,
sum(f.net_price) 
from gold.fact_sales f
join gold.dim_customer c
on f.customer_key = c.customer_key
group by c.market_segment
order by sum(f.net_price)

In [0]:
-- Q05 top 10 customer by net revenue
select 
  customer_key,
  round(sum(net_price), 2) as customer_revenue
from gold.fact_sales 
group by customer_key
order by customer_revenue desc

In [0]:
-- Q06 top 10 customer as a share of total_revenue
with total_revenue as (
  select sum(net_price) as grand_total
  from gold.fact_sales
)
select
  f.customer_key,
  round(sum(f.net_price), 2) as customer_revenue,
  round(sum(f.net_price) / t.grand_total * 100, 2) as revenue_share_pct
from gold.fact_sales f
cross join total_revenue t
group by f.customer_key, t.grand_total
order by customer_revenue desc
limit 10


In [0]:
-- Q07 top 10 products by net revenue
select 
part_key as produc_key,
sum(net_price) as product_revenue
from gold.fact_sales
group by part_key
order by product_revenue desc
limit 10

In [0]:
-- Q08 bottom 10 products by net revenue
select 
part_key as produc_key,
sum(net_price) as product_revenue
from gold.fact_sales
group by part_key
order by product_revenue
limit 10

In [0]:
-- Q09 revenue by product type
select
p.type as product_type,
sum(f.net_price) as product_revenue
from gold.fact_sales f
join gold.dim_product p
on f.part_key = p.product_key
group by p.type

In [0]:
-- Q10 top 10 suppliers by revenue
select
supplier_key,
sum(net_price) as supplier_revenue
from gold.fact_sales
group by supplier_key
order by supplier_revenue desc
limit 10

In [0]:
-- Q11 revenue share by top 10 suppliers
with total_revenue as (
  select sum(net_price) as grand_total
  from gold.fact_sales
)
select
  f.supplier_key,
  round(sum(f.net_price), 2) as customer_revenue,
  round(sum(f.net_price) / t.grand_total * 100, 2) as revenue_share_pct
from gold.fact_sales f
cross join total_revenue t
group by f.supplier_key, t.grand_total
order by customer_revenue desc
limit 10


In [0]:
-- Q12 revenue by customer region
select 
g.region_name as customer_region,
sum(f.net_price) as region_revenue
from gold.fact_sales f
join gold.dim_customer c
on f.customer_key = c.customer_key
join gold.dim_geography g
on c.nation_key = g.nation_key
group by customer_region

In [0]:
-- Q13 revenue by customer nation
select g.nation as customer_nation,
sum(f.net_price) as nation_revenue
from gold.fact_sales f
join gold.dim_customer c
on f.customer_key = c.customer_key
join gold.dim_geography g
on c.nation_key = g.nation_key
group by customer_nation
order by nation_revenue desc

In [0]:
-- Q14 revenue and volume split by return flag
select 
return_flag,
sum(net_price) as revenue,
sum(quantity) as volume
from gold.fact_sales
group by return_flag

In [0]:
-- Q15 Average discount by market segment
select 
c.market_segment,
avg(f.discount) as avg_discount
from gold.fact_sales f
join gold.dim_customer c
on f.customer_key = c.customer_key
group by c.market_segment

In [0]:
-- Q16 average ship to receipt days by shipping mode
select
ship_mode,
round(avg(datediff(receipt_date, ship_date)), 2) as avg_ship_to_receipt_days
from gold.fact_sales
group by ship_mode
order by avg_ship_to_receipt_days desc

In [0]:
-- Q17 order count and revenue by order priority
select
order_priority,
count(*) as order_count,
sum(net_price) as order_revenue
from gold.fact_sales
group by order_priority
order by order_count desc

In [0]:
-- Q18 monthly breakdown of order status
select
year(order_date) as year,
month(order_date) as month,
order_status,
count(distinct order_key) as order_count,
sum(net_price) as revenue
from gold.fact_sales
group by year, month, order_status
order by year, month, order_status

In [0]:
-- Q19 product revenue ranking within each market segment
select
market_segment,
part_key,
rank() over (partition by market_segment order by sum(net_price) desc) as revenue_rank
from gold.fact_sales
join gold.dim_product
on fact_sales.part_key = dim_product.product_key
join gold.dim_customer
on fact_sales.customer_key = dim_customer.customer_key
group by market_segment, part_key
order by market_segment, revenue_rank

In [0]:
-- Q20 revenue comparison of customer geography vs supplier geography by region
with customer_rev as (
  select
    g.region_name as region,
    sum(f.net_price) as customer_revenue
  from gold.fact_sales f
  join gold.dim_customer c
    on f.customer_key = c.customer_key
  join gold.dim_geography g
    on c.nation_key = g.nation_key
  group by g.region_name
),
supplier_rev as (
  select
    g.region_name as region,
    sum(f.net_price) as supplier_revenue
  from gold.fact_sales f
  join gold.dim_supplier s
    on f.supplier_key = s.supplier_key
  join gold.dim_geography g
    on s.nation_key = g.nation_key
  group by g.region_name
)
select
  coalesce(c.region, s.region) as region,
  round(c.customer_revenue, 2) as customer_revenue,
  round(s.supplier_revenue, 2) as supplier_revenue
from customer_rev c
full outer join supplier_rev s
  on c.region = s.region
order by region